# Injection and shift current: Figure 5

The magnetic injection current and the shift current of the four materials (TNN, GGA) at $B = 0$ and at $\mu_BB = 80$ meV, $\nu = 30$ meV. Here the field enters through the Zeeman Hamiltonian, the energy shift and the phase-space factor (no velocity shift). Each curve is the size $\|T\| = (\sum_{abc}T_{abc}^2)^{1/2}$ of the $C_3$-averaged tensor. Both currents are defined in the engine.

## Load engine

In [ ]:
%run engine.ipynb

## Settings

In [ ]:
N_ch = 240         # k-mesh over the zone (120 for a quick look)
n_ch = 1200        # photon energies (300 for a quick look)
nu = 0.03          # broadening (eV)

zone = slanted_grid(N_ch)

## Calculation

Photon energies from $0.9E_g$ to $1.4E_{\max}$. `eta_mag` (engine) gives both currents; the velocity shift is switched off here. About 15 min per material.

In [ ]:
for m in materials:
    file = f"data/channels_{m}_N{N_ch}_n{n_ch}.npz"
    if os.path.exists(file):
        continue
    a = params(m)["a"]
    omega = np.linspace(0.9 * gap(m), 1.4 * E_max(m), n_ch)
    out = {"omega": omega}
    for tag, B in [("B0", 0.0), ("Z", B80)]:
        eta, sigma = eta_mag(m, "TNN", B, omega, nu, zone, velocity_shift=False, shift=True)
        out["eta_" + tag] = np.abs(eta_SI(a)) * np.sqrt((eta**2).sum(axis=(1, 2, 3)))
        out["shift_" + tag] = shift_SI(a) * np.sqrt((sigma**2).sum(axis=(1, 2, 3)))
    np.savez(file, **out)

## Figure 5

Solid: $\mu_BB = 80$ meV, dotted: $B = 0$. The injection current is zero at $B = 0$ (time reversal), the shift current is not.

In [ ]:
def channels(m):
    d = np.load(f"data/channels_{m}_N{N_ch}_n{n_ch}.npz")
    tail = np.maximum.reduce([d["eta_Z"] / d["eta_Z"].max(), d["shift_Z"] / d["shift_Z"].max(), d["shift_B0"] / d["shift_B0"].max()])
    k = np.where(tail > 0.003)[0][-1] + 15
    print(f"{m}: largest ||eta|| = {d['eta_B0'].max():.1e} (B = 0), {d['eta_Z'].max():.3f} A m V^-2 s^-1 (80 meV); "
          f"largest ||sigma_shift|| = {d['shift_B0'].max():.1f}, {d['shift_Z'].max():.1f} nm uA V^-2")
    return d["omega"][:k], d["eta_B0"][:k], d["eta_Z"][:k], d["shift_B0"][:k], d["shift_Z"][:k]


fig = plt.figure(figsize=(TW, 5.6))
gs = fig.add_gridspec(3, 6, height_ratios=[1.35, 1, 1], hspace=0.55, wspace=1.1)
top = [fig.add_subplot(gs[0, 0:3]), fig.add_subplot(gs[0, 3:6])]
low = [[fig.add_subplot(gs[r, 2 * c:2 * c + 2]) for c in range(3)] for r in (1, 2)]
y_eta = r"$\|\eta^{a;(bc)}_{\mathrm{mag}}\|$ ($%s$)" % UNIT
y_shift = r"$\|\sigma^{a;(bc)}_{\mathrm{shift}}\|$ ($%s$)" % SUNIT

omega, eta_0, eta_Z, shift_0, shift_Z = channels("MoS2")
for ax, y0, yZ, ylabel in ((top[0], eta_0, eta_Z, y_eta), (top[1], shift_0, shift_Z, y_shift)):
    ax.plot(omega, y0, color=mat_color["MoS2"], lw=0.8, ls=":")
    ax.plot(omega, yZ, color=mat_color["MoS2"], lw=1.1)
    ax.set_ylabel(ylabel)
top[0].text(0.03, 0.9, mat_label["MoS2"], transform=top[0].transAxes, va="top")
top[1].legend(handles=[Line2D([], [], color="k", lw=1.1, label=r"$\mu_BB=80$ meV"),
                       Line2D([], [], color="k", lw=0.8, ls=":", label=r"$B=0$")], loc="upper left")
for c, m in enumerate(materials[1:]):
    omega, eta_0, eta_Z, shift_0, shift_Z = channels(m)
    for r, (y0, yZ) in enumerate(((eta_0, eta_Z), (shift_0, shift_Z))):
        ax = low[r][c]
        ax.plot(omega, y0, color=mat_color[m], lw=0.7, ls=":")
        ax.plot(omega, yZ, color=mat_color[m], lw=1.0)
        ax.yaxis.set_major_locator(MaxNLocator(4))
        ax.xaxis.set_major_locator(MaxNLocator(4))
        if r == 0:
            ax.text(0.05, 0.9, mat_label[m], transform=ax.transAxes, va="top")
low[0][0].set_ylabel(y_eta)
low[1][0].set_ylabel(y_shift)
for i, ax in enumerate(top + low[0] + low[1]):
    ax.set_xlabel(r"$\hbar\omega$ (eV)")
    ax.set_ylim(bottom=0)
    label(ax, f"({'abcdefgh'[i]})")
plt.savefig("figures/nonlinear_channels_all.pdf")
plt.show()